# RandomForest Cirrhosis Optimized
Target: Cirrhosis_Status

In [13]:
!pip install -q xgboost

In [33]:
# ==========================================================
# Liver Cirrhosis Prediction using Decision Tree
# ==========================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from sklearn.tree import DecisionTreeClassifier


# ==========================================================
# Load Dataset
# ==========================================================

df = pd.read_csv("/content/drive/MyDrive/cirrhosis.csv")

df.columns = df.columns.str.strip()


# Remove missing target values
df = df.dropna(subset=["Stage"])


print("="*60)
print("Dataset Shape :", df.shape)
print("="*60)



# ==========================================================
# Features & Target
# ==========================================================

TARGET = "Stage"


X = df.drop(
    columns=["ID", TARGET],
    errors="ignore"
)


y = df[TARGET].astype(int)



# Encode target

encoder = LabelEncoder()

y = encoder.fit_transform(y)


print("\nTarget Classes:")
print(encoder.classes_)



# ==========================================================
# Numerical & Categorical Features
# ==========================================================

num_cols = X.select_dtypes(
    include=["int64","float64"]
).columns


cat_cols = X.select_dtypes(
    include=["object"]
).columns



print("\nNumerical Features :", len(num_cols))
print("Categorical Features :", len(cat_cols))



# ==========================================================
# Preprocessing
# ==========================================================

numeric_transformer = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    )

])


categorical_transformer = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),

    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )

])


preprocessor = ColumnTransformer([

    (
        "num",
        numeric_transformer,
        num_cols
    ),

    (
        "cat",
        categorical_transformer,
        cat_cols
    )

])



# ==========================================================
# Train/Test Split
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)



# ==========================================================
# Decision Tree Model
# ==========================================================

dt = DecisionTreeClassifier(

    criterion="gini",

    max_depth=5,

    min_samples_split=10,

    min_samples_leaf=5,

    class_weight="balanced",

    random_state=42

)



# ==========================================================
# Pipeline
# ==========================================================

model = Pipeline([

    (
        "preprocessor",
        preprocessor
    ),

    (
        "classifier",
        dt
    )

])



# ==========================================================
# Training
# ==========================================================

print("\nTraining Decision Tree...\n")


model.fit(
    X_train,
    y_train
)



# ==========================================================
# Prediction
# ==========================================================

train_pred = model.predict(X_train)

test_pred = model.predict(X_test)



# ==========================================================
# Evaluation
# ==========================================================

print("="*60)


print(
    "Train Accuracy :",
    round(
        accuracy_score(
            y_train,
            train_pred
        )*100,
        2
    ),
    "%"
)


print(
    "Test Accuracy  :",
    round(
        accuracy_score(
            y_test,
            test_pred
        )*100,
        2
    ),
    "%"
)


print("="*60)



print("\nClassification Report\n")


print(
    classification_report(

        y_test,

        test_pred,

        target_names=[
            str(c)
            for c in encoder.classes_
        ]

    )
)



print("\nConfusion Matrix\n")


print(
    confusion_matrix(
        y_test,
        test_pred
    )
)



# ==========================================================
# Cross Validation
# ==========================================================

cv = StratifiedKFold(

    n_splits=5,

    shuffle=True,

    random_state=42

)


scores = cross_val_score(

    model,

    X,

    y,

    cv=cv,

    scoring="accuracy",

    n_jobs=-1

)



print("\nCross Validation Scores")

print(scores)



print(
    "\nMean Accuracy :",
    round(scores.mean()*100,2),
    "%"
)


print(
    "Standard Deviation :",
    round(scores.std()*100,2),
    "%"
)

Dataset Shape : (412, 20)

Target Classes:
[1 2 3 4]

Numerical Features : 11
Categorical Features : 7

Training Decision Tree...

Train Accuracy : 62.61 %
Test Accuracy  : 31.33 %

Classification Report

              precision    recall  f1-score   support

           1       0.09      0.25      0.13         4
           2       0.29      0.21      0.24        19
           3       0.33      0.45      0.38        31
           4       0.47      0.24      0.32        29

    accuracy                           0.31        83
   macro avg       0.29      0.29      0.27        83
weighted avg       0.35      0.31      0.31        83


Confusion Matrix

[[ 1  1  2  0]
 [ 3  4 11  1]
 [ 4  6 14  7]
 [ 3  3 16  7]]

Cross Validation Scores
[0.30120482 0.38554217 0.47560976 0.40243902 0.34146341]

Mean Accuracy : 38.13 %
Standard Deviation : 5.89 %


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
